# 🐶 Dog Emotion Classifier — CNN
**Introduction to Deep Learning | Final Project**

Run each cell from top to bottom.

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Step 2 — Import Libraries

In [ ]:
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

print("TensorFlow:", tf.__version__)

## Step 3 — Set Paths

In [ ]:
BASE_PATH  = "/content/drive/MyDrive/Dog_Emotions_Classification_Datasets"
TRAIN_PATH = os.path.join(BASE_PATH, "train")
VAL_PATH   = os.path.join(BASE_PATH, "val")
TEST_PATH  = os.path.join(BASE_PATH, "test")

IMG_SIZE   = (128, 128)
BATCH_SIZE = 16
EPOCHS     = 30

print("Train:", TRAIN_PATH)
print("Val  :", VAL_PATH)
print("Test :", TEST_PATH)

## Step 4 — Verify Dataset

In [ ]:
IMG_EXTS = (".jpg", ".jpeg", ".png", ".webp")
CLASS_NAMES = ["Angry", "Happy", "Sad"]

print(f"{"Split":<10} {"Class":<12} {"Count":>6}")
print("-" * 32)
grand_total = 0
for split, path in [("train", TRAIN_PATH), ("val", VAL_PATH), ("test", TEST_PATH)]:
    sub = 0
    for cls in CLASS_NAMES:
        folder = os.path.join(path, cls)
        count = len([f for f in os.listdir(folder) if f.lower().endswith(IMG_EXTS)]) if os.path.isdir(folder) else 0
        flag = "" if count >= 100 else " ⚠ need more"
        print(f"{split:<10} {cls:<12} {count:>6}{flag}")
        sub += count
    print(f"{"":<10} {"SUBTOTAL":<12} {sub:>6}")
    print()
    grand_total += sub
print(f"Grand total: {grand_total} images")

## Step 5 — Data Generators with Augmentation

**Improvements over original:**
- Added ,  for better generalization
- Validation and test use rescale only (no augmentation)

In [ ]:
# Training — with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    rotation_range=20,
    zoom_range=0.20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    brightness_range=[0.75, 1.25],
    fill_mode="reflect"
)

# Validation and test — rescale only
eval_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    TRAIN_PATH, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode="categorical"
)
val_data = eval_datagen.flow_from_directory(
    VAL_PATH, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode="categorical"
)
test_data = eval_datagen.flow_from_directory(
    TEST_PATH, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode="categorical",
    shuffle=False
)

CLASS_LABELS = list(train_data.class_indices.keys())
NUM_CLASSES  = train_data.num_classes
print("Classes     :", CLASS_LABELS)
print("Train       :", train_data.samples)
print("Validation  :", val_data.samples)
print("Test        :", test_data.samples)

## Step 6 — Class Weights

Fixes the imbalance between Angry (fewer images) and Happy (more images).

In [ ]:
cw = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_data.classes),
    y=train_data.classes
)
class_weight_dict = dict(enumerate(cw))

for cls, idx in train_data.class_indices.items():
    print(f"{cls}: weight = {cw[idx]:.3f}")

## Step 7 — Build the CNN Model

**Improvements over original:**
- Added 2nd and 3rd Conv2D block
- BatchNormalization after each block
- GlobalAveragePooling instead of Flatten
- Dropout(0.4) to prevent overfitting

In [ ]:
model = models.Sequential([

    # Block 1
    layers.Conv2D(32, (3, 3), activation="relu",
                  input_shape=(128, 128, 3), padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Block 2
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Block 3
    layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Classifier
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(NUM_CLASSES, activation="softmax")
])

model.summary()

## Step 8 — Compile

In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

## Step 9 — Callbacks

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=7,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5,
        patience=3, min_lr=1e-6, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "best_dog_model.h5",
        monitor="val_accuracy",
        save_best_only=True, verbose=1
    )
]

## Step 10 — Train Model

In [ ]:
history = model.fit(
    train_data,
    epochs=EPOCHS,
    validation_data=val_data,
    class_weight=class_weight_dict,
    callbacks=callbacks
)

## Step 11 — Plot Accuracy and Loss

Both curves are required — accuracy shows learning progress, loss shows overfitting.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history["accuracy"],     label="Train")
ax1.plot(history.history["val_accuracy"], label="Validation")
ax1.set_title("Model Accuracy per Epoch")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Accuracy")
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history.history["loss"],         label="Train")
ax2.plot(history.history["val_loss"],     label="Validation")
ax2.set_title("Model Loss per Epoch")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss")
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle("Training Results — Dog Emotion Classifier CNN", fontweight="bold")
plt.tight_layout()
plt.savefig("training_results.png", dpi=150)
plt.show()

## Step 12 — Evaluate on Test Set

Always evaluate on the held-out **test** set — never the validation set.

In [ ]:
best_model = tf.keras.models.load_model("best_dog_model.h5")

test_loss, test_acc = best_model.evaluate(test_data, verbose=0)
print(f"Test Accuracy : {test_acc:.2%}")
print(f"Test Loss     : {test_loss:.4f}")

## Step 13 — Confusion Matrix and Per-Class Report

In [ ]:
test_data.reset()
y_pred = np.argmax(best_model.predict(test_data, verbose=0), axis=1)
y_true = test_data.classes

print(classification_report(y_true, y_pred, target_names=CLASS_LABELS))

ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred,
    display_labels=CLASS_LABELS,
    cmap="Blues", normalize="true"
)
plt.title("Confusion Matrix (row-normalized)")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

## Step 14 — Save Model and Class Labels

Download both files — you need them for the Streamlit app.

In [ ]:
best_model.save("dog_emotion_model.h5")

with open("class_labels.json", "w") as f:
    json.dump(CLASS_LABELS, f)

print("Saved: dog_emotion_model.h5")
print("Class labels:", CLASS_LABELS)

from google.colab import files
files.download("dog_emotion_model.h5")
files.download("class_labels.json")